# Signal Ranker — Sandbox Demo
**Redrob Intelligent Candidate Discovery & Ranking Challenge**

This notebook runs the Signal ranker on a sample candidate pool and produces a ranked CSV output.

- No GPU required
- No external API calls
- Runs in < 30 seconds on CPU

**Instructions:** Run all cells top to bottom (Runtime → Run all)

## Step 1 — Load the ranker code

In [ ]:
# Download rank.py from GitHub
!wget -q https://raw.githubusercontent.com/kakarot077/signal-ai-recruiter/main/rank.py
print('rank.py loaded ✓')

## Step 2 — Load sample candidates
We use the 50-candidate sample provided in the challenge bundle.
You can also upload your own `candidates.jsonl` file below.

In [ ]:
import json

# Option A: Use built-in sample data (50 candidates from challenge bundle)
# Option B: Upload your own file — uncomment the block below

# --- Option B: Upload your own candidates.jsonl ---
# from google.colab import files
# uploaded = files.upload()  # upload candidates.jsonl or sample_candidates.json

# Download sample_candidates.json from GitHub
!wget -q https://raw.githubusercontent.com/kakarot077/signal-ai-recruiter/main/sample_candidates.json -O sample_candidates.json

# Convert sample_candidates.json (array format) to JSONL format for rank.py
with open('sample_candidates.json') as f:
    candidates = json.load(f)

with open('sample_candidates.jsonl', 'w') as f:
    for c in candidates:
        f.write(json.dumps(c) + '\n')

print(f'Loaded {len(candidates)} sample candidates ✓')
print(f'Sample IDs: {[c["candidate_id"] for c in candidates[:5]]}')

## Step 3 — Run the ranker

In [ ]:
import subprocess, time

start = time.time()

result = subprocess.run(
    ['python', 'rank.py',
     '--candidates', 'sample_candidates.jsonl',
     '--out', 'sample_submission.csv',
     '--top', '20',   # top 20 from 50-candidate sample
     '--debug'],
    capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)
print(f'\nCompleted in {time.time()-start:.1f}s')

## Step 4 — View ranked output

In [ ]:
import pandas as pd

df = pd.read_csv('sample_submission.csv')
print(f'Ranked {len(df)} candidates\n')
pd.set_option('display.max_colwidth', 120)
print(df.to_string(index=False))

## Step 5 — Score breakdown visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

ranks = df['rank'].tolist()
scores = df['score'].tolist()
ids = df['candidate_id'].tolist()

bars = ax.barh([f"#{r} {cid}" for r, cid in zip(ranks, ids)],
               scores, color='#00D4FF', alpha=0.8, edgecolor='#0A2A3A')

ax.set_xlabel('Signal Score', fontsize=12)
ax.set_title('Signal Ranker — Top Candidates by Score', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.0)
ax.invert_yaxis()
ax.set_facecolor('#080C14')
fig.patch.set_facecolor('#0D1422')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.title.set_color('#00D4FF')
ax.spines['bottom'].set_color('#1C2A42')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#1C2A42')

for bar, score in zip(bars, scores):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=8, color='#00D4FF')

plt.tight_layout()
plt.savefig('ranking_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✓')

## Step 6 — Download output CSV

In [ ]:
from google.colab import files
files.download('sample_submission.csv')
print('Download started ✓')

---
## About Signal Ranker

**Architecture:** 5-layer deterministic scoring pipeline

| Layer | Weight | What it measures |
|---|---|---|
| Skill Match | 35% | Production embedding/retrieval experience |
| Career Trajectory | 25% | Product vs services, title fit, stability |
| Behavioral Signals | 20% | 23 Redrob signals — availability, engagement |
| Location Fit | 12% | Pune/Noida/Tier-1 India preference |
| Experience Band | 8% | 5–9 years per JD |

**No API calls. No GPU. No external models. Pure Python stdlib.**

GitHub: https://github.com/kakarot077/signal-ai-recruiter

Reproduce full submission:
```bash
python rank.py --candidates ./candidates.jsonl --out ./submission.csv
```